# 06 检索系统开发第二部分 — 检索流水线演示

> **英文 query 优先**。按 schedule 分步展示：环境 → 查询增强 → BM25 → 向量单路 → 双路诊断 → **多路融合** → 导出。
>
> **内核**：`med-rag-verify` · 从本 notebook 所在目录打开，按 **C0 → C8** 顺序运行。

| 章节 | 对应阶段 | 内容 |
|------|----------|------|
| **C0** | 阶段 0 | 环境与路径检查（chunks / slim / Chroma） |
| **C1** | 05 联调 | 查询增强 `EnhancedQuery` |
| **C2** | 阶段 1 | BM25 建索引与分词 |
| **C3** | 阶段 1 | BM25 检索结果（schedule 验证 query） |
| **C4** | 阶段 2 · 向量路 | 向量单路 smoke（融合前的语义腿验证） |
| **C5** | 阶段 2 · 诊断 | 向量 vs BM25 原始 Top-K 重叠（未融合） |
| **C6** | 阶段 2 | `MultiPathRetriever` 默认 RRF 融合结果 |
| **C7** | 阶段 2 | 三种融合策略（simple / rrf / weighted）并排对比 |
| **C8** | 导出 | 写入 `outputs/samples/*.json` |

阶段 3（重排）/ 4（完整 pipeline）待实现后追加 C9+。

## C0 环境与路径

挂载 Stage04/05/06 的 `sys.path`，检查样本库、slim 回查文件、BM25 语料是否存在。

In [1]:
import json
import sys
import time
from pathlib import Path

STAGE06 = Path("..").resolve()
STAGE05 = (STAGE06.parent / "05 检索系统开发第一部分").resolve()
STAGE04 = (STAGE06.parent / "04 向量化与索引构建").resolve()
SRC06 = STAGE06 / "src"
SRC05 = STAGE05 / "src"
SRC04 = STAGE04 / "src"

for p in (SRC06, SRC05, SRC04):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from config import (
    COLLECTION_SAMPLE,
    EMBED_MODEL,
    RERANK_MODEL,
    resolve_chroma,
    resolve_chunks_path,
    resolve_slim_path,
)

MODE = "sample"  # sample | full
CHUNKS_PATH = resolve_chunks_path(MODE)
CHROMA_DIR, COLLECTION = resolve_chroma(MODE)
SLIM_PATH = resolve_slim_path()

env_report = {
    "stage06": str(STAGE06),
    "mode": MODE,
    "chunks_path": str(CHUNKS_PATH),
    "chunks_exists": CHUNKS_PATH.is_file(),
    "slim_path": str(SLIM_PATH),
    "slim_exists": SLIM_PATH.is_file(),
    "chroma_dir": str(CHROMA_DIR),
    "chroma_exists": CHROMA_DIR.is_dir(),
    "collection": COLLECTION,
    "embed_model": EMBED_MODEL,
    "rerank_model": RERANK_MODEL,
}
print(json.dumps(env_report, indent=2, ensure_ascii=False))

{
  "stage06": "D:\\谷歌\\06 检索系统开发第二部分",
  "mode": "sample",
  "chunks_path": "D:\\谷歌\\03 文档解析与分割\\data\\processed\\chunks_sample.jsonl",
  "chunks_exists": true,
  "slim_path": "D:\\谷歌\\06 检索系统开发第二部分\\data\\oa_comm_slim.jsonl",
  "slim_exists": true,
  "chroma_dir": "D:\\谷歌\\04 向量化与索引构建\\data\\chroma_db",
  "chroma_exists": true,
  "collection": "pmc_oa_comm_sample",
  "embed_model": "BAAI/bge-small-en-v1.5",
  "rerank_model": "BAAI/bge-reranker-base"
}


## C1 查询增强（05 模块）

加载 `MedicalQueryEnhancer`，对 schedule 首批验证 query 生成 `vector_query` / `keyword_query` / `filters`。

In [2]:
from query_enhancer import MedicalQueryEnhancer

DEMO_QUERIES = [
    "metformin cardiovascular effects",
    "papers on malaria after 2015",
    "MI treatment guideline",
    "circadian rhythm sliding window chunks",
    "warfarin atrial fibrillation elderly",
]

enhancer = MedicalQueryEnhancer(STAGE05 / "data" / "medical_synonyms.json")
enhanced_list = [enhancer.process(q) for q in DEMO_QUERIES]

for eq in enhanced_list:
    print("\n===", eq.original)
    print("  vector :", eq.vector_query)
    print("  keyword:", eq.keyword_query)
    print("  expanded:", eq.expanded_terms)
    print("  filters:", [(f.key, f.value, f.executable) for f in eq.filters])
    print("  chroma_where:", eq.chroma_where())


=== metformin cardiovascular effects
  vector : metformin cardiovascular effects
  keyword: metformin cardiovascular effects biguanide antidiabetic heart disease metformin cardiovascular effects cardiovascular outcomes metformin
  expanded: ['metformin', 'biguanide antidiabetic', 'cardiovascular', 'heart disease', 'metformin cardiovascular effects', 'cardiovascular outcomes metformin']
  filters: []
  chroma_where: None

=== papers on malaria after 2015
  vector : papers on malaria after 2015
  keyword: papers malaria after 2015 plasmodium
  expanded: ['malaria', 'plasmodium']
  filters: [('year_gte', 2015, False)]
  chroma_where: None

=== MI treatment guideline
  vector : MI treatment guideline
  keyword: mi treatment guideline myocardial infarction heart attack
  expanded: ['myocardial infarction', 'heart attack']
  filters: []
  chroma_where: None

=== circadian rhythm sliding window chunks
  vector : circadian rhythm sliding window chunks
  keyword: circadian rhythm sliding windo

## C2 BM25 建索引（阶段 1）

从 chunks JSONL 构建内存 BM25 索引，展示语料规模与分词示例。

In [3]:
from bm25_index import BM25Index, tokenize

bm25 = BM25Index()
t0 = time.perf_counter()
n_chunks = bm25.build(mode=MODE)
build_sec = time.perf_counter() - t0

sample_text = "The Malaria vaccine efficacy in 2015 for Plasmodium falciparum"
sample_tokens = tokenize(sample_text)

bm25_build_report = {
    "source_path": str(bm25.source_path),
    "chunks_indexed": n_chunks,
    "build_sec": round(build_sec, 4),
    "tokenize_example": {
        "text": sample_text,
        "tokens": sample_tokens,
    },
}
print(json.dumps(bm25_build_report, indent=2, ensure_ascii=False))

{
  "source_path": "D:\\谷歌\\03 文档解析与分割\\data\\processed\\chunks_sample.jsonl",
  "chunks_indexed": 1267,
  "build_sec": 0.1015,
  "tokenize_example": {
    "text": "The Malaria vaccine efficacy in 2015 for Plasmodium falciparum",
    "tokens": [
      "malaria",
      "vaccine",
      "efficacy",
      "2015",
      "plasmodium",
      "falciparum"
    ]
  }
}


## C3 BM25 检索结果（阶段 1）

对每条增强 query 的 `keyword_query` 执行 BM25 `top_k=5`，表格化展示 rank / score / chunk_id / title。

In [4]:
import pandas as pd

TOP_K_BM25 = 5
bm25_results_by_query = {}
rows = []

for eq in enhanced_list:
    t0 = time.perf_counter()
    hits = bm25.search(eq.keyword_query, top_k=TOP_K_BM25)
    elapsed = time.perf_counter() - t0
    bm25_results_by_query[eq.original] = hits
    for h in hits:
        rows.append({
            "query": eq.original,
            "keyword_query": eq.keyword_query,
            "rank": h["rank"],
            "score": round(h["score"], 4),
            "chunk_id": h["chunk_id"],
            "doc_id": h.get("doc_id"),
            "strategy": h.get("strategy"),
            "title": (h.get("source_title") or "")[:80],
            "latency_sec": round(elapsed, 4),
        })

df_bm25 = pd.DataFrame(rows)
display(df_bm25)
print(f"共 {len(enhanced_list)} 条 query，BM25 总命中行数: {len(df_bm25)}")

,query,keyword_query,rank,score,chunk_id,doc_id,strategy,title,latency_sec
0,metformin cardiovascular effects,metformin cardiovascular effects biguanide ant...,1,26.8930,PMC521687,PMC521687,single,Factors influencing preoperative stress respon...,0.0029
1,metformin cardiovascular effects,metformin cardiovascular effects biguanide ant...,2,25.2508,PMC521493,PMC521493,single,Lycopene from two food sources does not affect...,0.0029
2,metformin cardiovascular effects,metformin cardiovascular effects biguanide ant...,3,18.9435,PMC520826,PMC520826,single,Daily rhythms in plasma levels of homocysteine,0.0029
3,metformin cardiovascular effects,metformin cardiovascular effects biguanide ant...,4,18.2745,PMC523844,PMC523844,single,Distribution of Major Health Risks: Findings f...,0.0029
4,metformin cardiovascular effects,metformin cardiovascular effects biguanide ant...,5,16.0497,PMC518966,PMC518966,single,What is the impact of the ACE gene insertion/d...,0.0029
5,papers on malaria after 2015,papers malaria after 2015 plasmodium,1,15.1072,PMC514496,PMC514496,single,Malaria morbidity and immunity among residents...,0.0010
6,papers on malaria after 2015,papers malaria after 2015 plasmodium,2,14.4909,PMC517943,PMC517943,single,Antibodies from malaria-exposed pregnant women...,0.0010
7,papers on malaria after 2015,papers malaria after 2015 plasmodium,3,12.7054,PMC517944,PMC517944,single,Competitive release of drug resistance followi...,0.0010
8,papers on malaria after 2015,papers malaria after 2015 plasmodium,4,12.4449,PMC176545,PMC176545,single,The Transcriptome of the Intraerythrocytic Dev...,0.0010
9,papers on malaria after 2015,papers malaria after 2015 plasmodium,5,12.2820,PMC434153,PMC434153,single,Immunity Promotes Virulence Evolution in a Mal...,0.0010


共 5 条 query，BM25 总命中行数: 25


## C4 向量单路检索 smoke（阶段 2 · 向量腿）

**用途（在融合之前单独跑通向量路）**：

1. 验证 04 Chroma + 05 `vector_query` 能正常返回语义 Top-K
2. 确认 `chroma_where()` 过滤（如 `strategy=sliding_window`）是否可用
3. 为 C7/C8 的 `MultiPathRetriever` 提供可对照的「向量路基准结果」

> C4 不是最终交付形态，而是多路检索中**语义腿**的独立 smoke test；融合在 C7 完成。

In [5]:
from embedder import DocumentEmbedder
from index_builder import ChromaIndexBuilder, count_embeddings_sqlite

embedder = DocumentEmbedder(model_name=EMBED_MODEL)
builder = ChromaIndexBuilder(str(CHROMA_DIR), COLLECTION, embedder)
n_chroma = count_embeddings_sqlite(CHROMA_DIR)
print(f"Chroma 条数: {n_chroma}, collection: {COLLECTION}")

TOP_K_VECTOR = 5
vector_results_by_query = {}
vec_rows = []

for eq in enhanced_list:
    where = eq.chroma_where()
    t0 = time.perf_counter()
    res = builder.query(eq.vector_query, n_results=TOP_K_VECTOR, where_filter=where)
    elapsed = time.perf_counter() - t0
    ids = res["ids"][0]
    dists = res["distances"][0]
    metas = res["metadatas"][0]
    docs = res["documents"][0]
    hits = []
    for rank, (cid, dist, meta, doc) in enumerate(zip(ids, dists, metas, docs), start=1):
        hit = {
            "chunk_id": cid,
            "doc_id": meta.get("doc_id"),
            "source_title": meta.get("source_title"),
            "strategy": meta.get("strategy"),
            "text": doc,
            "source": "vector",
            "score": float(1.0 - dist),  # cosine distance → 相似度示意
            "distance": float(dist),
            "rank": rank,
        }
        hits.append(hit)
        vec_rows.append({
            "query": eq.original,
            "vector_query": eq.vector_query,
            "where": where,
            "rank": rank,
            "distance": round(dist, 4),
            "score": round(1.0 - dist, 4),
            "chunk_id": cid,
            "doc_id": meta.get("doc_id"),
            "strategy": meta.get("strategy"),
            "title": (meta.get("source_title") or "")[:80],
            "latency_sec": round(elapsed, 4),
        })
    vector_results_by_query[eq.original] = hits

df_vector = pd.DataFrame(vec_rows)
display(df_vector)

Chroma 条数: 1267, collection: pmc_oa_comm_sample
  [query] Chroma where= 失败，改用 over-fetch + Python 过滤 (InternalError)


,query,vector_query,where,rank,distance,score,chunk_id,doc_id,strategy,title,latency_sec
0,metformin cardiovascular effects,metformin cardiovascular effects,None,1,0.3709,0.6291,PMC523838_chunk2,PMC523838,sliding_window,Nevirapine and Efavirenz Elicit Different Chan...,5.5296
1,metformin cardiovascular effects,metformin cardiovascular effects,None,2,0.3861,0.6139,PMC524031,PMC524031,single,Metabolic response of people with type 2 diabe...,5.5296
2,metformin cardiovascular effects,metformin cardiovascular effects,None,3,0.3979,0.6021,PMC524175,PMC524175,single,Single nucleotide polymorphisms in the apolipo...,5.5296
3,metformin cardiovascular effects,metformin cardiovascular effects,None,4,0.4039,0.5961,PMC524029,PMC524029,single,β3-adrenoceptor agonist prevents alterations o...,5.5296
4,metformin cardiovascular effects,metformin cardiovascular effects,None,5,0.4113,0.5887,PMC524503,PMC524503,single,Association of changes in health-related quali...,5.5296
5,papers on malaria after 2015,papers on malaria after 2015,None,1,0.3684,0.6316,PMC523837,PMC523837,single,Assessment of Volume Depletion in Children wit...,0.0089
6,papers on malaria after 2015,papers on malaria after 2015,None,2,0.3772,0.6228,PMC522803,PMC522803,single,Exposure to malaria affects the regression of ...,0.0089
7,papers on malaria after 2015,papers on malaria after 2015,None,3,0.4009,0.5991,PMC524252,PMC524252,single,The Risk of a Mosquito-Borne Infectionin a Het...,0.0089
8,papers on malaria after 2015,papers on malaria after 2015,None,4,0.4112,0.5888,PMC524359,PMC524359,single,Under-representation of developing countries i...,0.0089
9,papers on malaria after 2015,papers on malaria after 2015,None,5,0.4134,0.5866,PMC524376_chunk0,PMC524376,sliding_window,"A Randomised, Double-Blind, Controlled Vaccine...",0.0089


## C5 向量 vs BM25 诊断对比（融合前）

并列展示同 query 下两路 **原始** Top-K 的 `chunk_id` 重叠——用于观察「语义路 vs 关键词路」差异，**不是**融合结果。融合见 C7/C8。

In [6]:
COMPARE_QUERY = DEMO_QUERIES[0]  # metformin cardiovascular effects
eq = next(e for e in enhanced_list if e.original == COMPARE_QUERY)
vec_ids = [h["chunk_id"] for h in vector_results_by_query[COMPARE_QUERY]]
bm25_ids = [h["chunk_id"] for h in bm25_results_by_query[COMPARE_QUERY]]
overlap = sorted(set(vec_ids) & set(bm25_ids))

compare_report = {
    "query": COMPARE_QUERY,
    "vector_query": eq.vector_query,
    "keyword_query": eq.keyword_query,
    "vector_top5": vec_ids,
    "bm25_top5": bm25_ids,
    "overlap_chunk_ids": overlap,
    "overlap_count": len(overlap),
    "note": "原始双路对比（未融合）；融合结果见 C6/C7",
}
print(json.dumps(compare_report, indent=2, ensure_ascii=False))

side_by_side = pd.DataFrame({
    "vector_rank": range(1, len(vec_ids) + 1),
    "vector_chunk_id": vec_ids,
    "bm25_rank": range(1, len(bm25_ids) + 1),
    "bm25_chunk_id": bm25_ids,
})
display(side_by_side)

{
  "query": "metformin cardiovascular effects",
  "vector_query": "metformin cardiovascular effects",
  "keyword_query": "metformin cardiovascular effects biguanide antidiabetic heart disease metformin cardiovascular effects cardiovascular outcomes metformin",
  "vector_top5": [
    "PMC523838_chunk2",
    "PMC524031",
    "PMC524175",
    "PMC524029",
    "PMC524503"
  ],
  "bm25_top5": [
    "PMC521687",
    "PMC521493",
    "PMC520826",
    "PMC523844",
    "PMC518966"
  ],
  "overlap_chunk_ids": [],
  "overlap_count": 0,
  "note": "原始双路对比（未融合）；融合结果见 C6/C7"
}


,vector_rank,vector_chunk_id,bm25_rank,bm25_chunk_id
0,1,PMC523838_chunk2,1,PMC521687
1,2,PMC524031,2,PMC521493
2,3,PMC524175,3,PMC520826
3,4,PMC524029,4,PMC523844
4,5,PMC524503,5,PMC518966


## C6 多路召回与融合（阶段 2）

使用 `MultiPathRetriever`：向量路 + BM25 路 → 默认 **RRF** 融合，展示融合后 Top-K。

In [7]:
from embedder import DocumentEmbedder
from index_builder import ChromaIndexBuilder
from multipath_retriever import MultiPathRetriever
from config import DEFAULT_FUSION_STRATEGY

embedder = DocumentEmbedder(model_name=EMBED_MODEL)
builder = ChromaIndexBuilder(str(CHROMA_DIR), COLLECTION, embedder)
retriever = MultiPathRetriever(builder, bm25, mode=MODE)

TOP_K_VECTOR = 10
TOP_K_KEYWORD = 10
TOP_K_FUSED = 5

fusion_results_by_query = {}
fusion_rows = []

for eq in enhanced_list:
    t0 = time.perf_counter()
    result = retriever.retrieve(
        eq,
        top_k_vector=TOP_K_VECTOR,
        top_k_keyword=TOP_K_KEYWORD,
        fusion_strategy=DEFAULT_FUSION_STRATEGY,
        top_k_fused=TOP_K_FUSED,
    )
    elapsed = time.perf_counter() - t0
    fusion_results_by_query[eq.original] = result
    for h in result["fused"]:
        fusion_rows.append({
            "query": eq.original,
            "strategy": h["fusion_strategy"],
            "rank": h["rank"],
            "fusion_score": round(h["fusion_score"], 6) if h.get("fusion_score") is not None else None,
            "vector_rank": h.get("vector_rank"),
            "bm25_rank": h.get("bm25_rank"),
            "chunk_id": h["chunk_id"],
            "title": (h.get("source_title") or "")[:80],
            "latency_sec": round(elapsed, 4),
        })

df_fusion = pd.DataFrame(fusion_rows)
display(df_fusion)

  [query] Chroma where= 失败，改用 over-fetch + Python 过滤 (InternalError)


,query,strategy,rank,fusion_score,vector_rank,bm25_rank,chunk_id,title,latency_sec
0,metformin cardiovascular effects,rrf,1,0.016393,1.0,NaN,PMC523838_chunk2,Nevirapine and Efavirenz Elicit Different Chan...,1.6171
1,metformin cardiovascular effects,rrf,2,0.016393,NaN,1.0,PMC521687,Factors influencing preoperative stress respon...,1.6171
2,metformin cardiovascular effects,rrf,3,0.016129,2.0,NaN,PMC524031,Metabolic response of people with type 2 diabe...,1.6171
3,metformin cardiovascular effects,rrf,4,0.016129,NaN,2.0,PMC521493,Lycopene from two food sources does not affect...,1.6171
4,metformin cardiovascular effects,rrf,5,0.015873,3.0,NaN,PMC524175,Single nucleotide polymorphisms in the apolipo...,1.6171
5,papers on malaria after 2015,rrf,1,0.030835,2.0,8.0,PMC522803,Exposure to malaria affects the regression of ...,0.0163
6,papers on malaria after 2015,rrf,2,0.030679,1.0,10.0,PMC523837,Assessment of Volume Depletion in Children wit...,0.0163
7,papers on malaria after 2015,rrf,3,0.016393,NaN,1.0,PMC514496,Malaria morbidity and immunity among residents...,0.0163
8,papers on malaria after 2015,rrf,4,0.016129,NaN,2.0,PMC517943,Antibodies from malaria-exposed pregnant women...,0.0163
9,papers on malaria after 2015,rrf,5,0.015873,3.0,NaN,PMC524252,The Risk of a Mosquito-Borne Infectionin a Het...,0.0163


## C7 三种融合策略并排对比

对同一条 query 分别运行 `simple` / `rrf` / `weighted`，对比融合后 Top-5 的 `chunk_id` 与 `fusion_score`。

In [8]:
FUSION_STRATEGIES = ["simple", "rrf", "weighted"]
COMPARE_QUERY = DEMO_QUERIES[0]
eq_cmp = next(e for e in enhanced_list if e.original == COMPARE_QUERY)

strategy_tables = {}
for strat in FUSION_STRATEGIES:
    r = retriever.retrieve(
        eq_cmp,
        top_k_vector=TOP_K_VECTOR,
        top_k_keyword=TOP_K_KEYWORD,
        fusion_strategy=strat,
        top_k_fused=TOP_K_FUSED,
    )
    strategy_tables[strat] = r["fused"]

cmp_rows = []
for strat, hits in strategy_tables.items():
    for h in hits:
        cmp_rows.append({
            "fusion_strategy": strat,
            "rank": h["rank"],
            "chunk_id": h["chunk_id"],
            "fusion_score": h.get("fusion_score"),
            "vector_rank": h.get("vector_rank"),
            "bm25_rank": h.get("bm25_rank"),
        })

df_strategy_cmp = pd.DataFrame(cmp_rows)
display(df_strategy_cmp)

fusion_strategy_report = {
    "query": COMPARE_QUERY,
    "strategies": {
        s: [h["chunk_id"] for h in strategy_tables[s]] for s in FUSION_STRATEGIES
    },
    "default_strategy": DEFAULT_FUSION_STRATEGY,
}
print(json.dumps(fusion_strategy_report, indent=2, ensure_ascii=False))

,fusion_strategy,rank,chunk_id,fusion_score,vector_rank,bm25_rank
0,simple,1,PMC523838_chunk2,NaN,1.0,NaN
1,simple,2,PMC524031,NaN,2.0,NaN
2,simple,3,PMC524175,NaN,3.0,NaN
3,simple,4,PMC524029,NaN,4.0,NaN
4,simple,5,PMC524503,NaN,5.0,NaN
5,rrf,1,PMC523838_chunk2,0.016393,1.0,NaN
6,rrf,2,PMC521687,0.016393,NaN,1.0
7,rrf,3,PMC524031,0.016129,2.0,NaN
8,rrf,4,PMC521493,0.016129,NaN,2.0
9,rrf,5,PMC524175,0.015873,3.0,NaN


{
  "query": "metformin cardiovascular effects",
  "strategies": {
    "simple": [
      "PMC523838_chunk2",
      "PMC524031",
      "PMC524175",
      "PMC524029",
      "PMC524503"
    ],
    "rrf": [
      "PMC523838_chunk2",
      "PMC521687",
      "PMC524031",
      "PMC521493",
      "PMC524175"
    ],
    "weighted": [
      "PMC523838_chunk2",
      "PMC524031",
      "PMC521687",
      "PMC521493",
      "PMC524175"
    ]
  },
  "default_strategy": "rrf"
}


## C8 导出样例

将 BM25、向量 smoke、双路诊断、融合结果写入 `outputs/samples/`。

In [9]:
out_dir = STAGE06 / "outputs" / "samples"
out_dir.mkdir(parents=True, exist_ok=True)

bm25_export = {
    "mode": MODE,
    "chunks_path": str(CHUNKS_PATH),
    "chunks_indexed": n_chunks,
    "top_k": TOP_K_BM25,
    "queries": [
        {
            "original": eq.original,
            "keyword_query": eq.keyword_query,
            "enhanced": eq.to_dict(),
            "hits": bm25_results_by_query[eq.original],
        }
        for eq in enhanced_list
    ],
}

vector_export = {
    "mode": MODE,
    "chroma_dir": str(CHROMA_DIR),
    "collection": COLLECTION,
    "top_k": TOP_K_VECTOR,
    "queries": [
        {
            "original": eq.original,
            "vector_query": eq.vector_query,
            "chroma_where": eq.chroma_where(),
            "hits": vector_results_by_query[eq.original],
        }
        for eq in enhanced_list
    ],
}

paths = {
    "bm25_examples": out_dir / "bm25_examples.json",
    "vector_smoke": out_dir / "vector_smoke_sample.json",
    "retrieval_compare": out_dir / "retrieval_compare.json",
    "fusion_examples": out_dir / "fusion_examples.json",
}

fusion_export = {
    "mode": MODE,
    "default_strategy": DEFAULT_FUSION_STRATEGY,
    "top_k_fused": TOP_K_FUSED,
    "queries": [
        {
            "original": eq.original,
            "fusion_strategy": DEFAULT_FUSION_STRATEGY,
            "vector_hits": fusion_results_by_query[eq.original]["vector_hits"],
            "keyword_hits": fusion_results_by_query[eq.original]["keyword_hits"],
            "fused": fusion_results_by_query[eq.original]["fused"],
        }
        for eq in enhanced_list
    ],
    "strategy_compare": fusion_strategy_report,
}

with open(paths["bm25_examples"], "w", encoding="utf-8") as f:
    json.dump(bm25_export, f, indent=2, ensure_ascii=False)
with open(paths["vector_smoke"], "w", encoding="utf-8") as f:
    json.dump(vector_export, f, indent=2, ensure_ascii=False)
with open(paths["retrieval_compare"], "w", encoding="utf-8") as f:
    json.dump(compare_report, f, indent=2, ensure_ascii=False)
with open(paths["fusion_examples"], "w", encoding="utf-8") as f:
    json.dump(fusion_export, f, indent=2, ensure_ascii=False)

print("已导出:")
for k, p in paths.items():
    print(f"  {k}: {p}")

已导出:
  bm25_examples: D:\谷歌\06 检索系统开发第二部分\outputs\samples\bm25_examples.json
  vector_smoke: D:\谷歌\06 检索系统开发第二部分\outputs\samples\vector_smoke_sample.json
  retrieval_compare: D:\谷歌\06 检索系统开发第二部分\outputs\samples\retrieval_compare.json
  fusion_examples: D:\谷歌\06 检索系统开发第二部分\outputs\samples\fusion_examples.json
